<div align="right" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/ExploreAI_logos/Logo blue_dark.png"  style="width:25px" align="right";/>
</div>

<div align="right" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img
 src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/alx-courses/aice/assets/Content_page_banner_blue_dots.png"
 alt="ALX Content Header"
 class="full-width-image"
/>
</div>

# SQL window functions

In this exercise, we will test our understanding and application of SQL window functions on a sample SQLite database file for a retail company called Northwind by performing complex calculations and analyses like ranking, running totals, and date differences. Ensure that you have downloaded the database file, Northwind.db.

## Learning objectives

By the end of this train, you should:
- Use the RANK() function to assign a ranking number to each row based on the order specified within the window.
- Use aggregate window functions to calculate running totals. 
- Use the LAG() function to help calculate the difference, in days, between consecutive date readings in our dataset.
- Use aggregate window functions to calculate the moving average. 

First, let's load our sample database:

In [1]:
# Load and activate the SQL extension to allow us to execute SQL in a Jupyter notebook.
%load_ext sql


In [2]:
# Load the Northwind database stored in your local machine. 
# Make sure the file is saved in the same folder as this notebook.
%sql sqlite:///Northwind.db
    

'Connected: @Northwind.db'

In [3]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

Here is a view of all of our tables in the database:

<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/Northwind_ERD.png"  style="width:500px";/>
<br>
<br>
    <em>Figure 1: Northwind ERD</em>
</div>

## Exercise

Run the necessary queries that will provide us with the following information. Compare your queries with the solutions at the end of this notebook.

### Exercise 1

Rank all the orders of a specific customer from the most recent to the least recent using window functions. Assume that the customer ID is `'ALFKI'`.

In [4]:
%%sql

SELECT
    OrderID,
    CustomerID,
    OrderDate,
    RANK() OVER (
        ORDER BY OrderDate DESC
    ) AS Ranking_orders
FROM
    Orders
WHERE
    CustomerID = "ALFKI"
LIMIT 50;

 * sqlite:///Northwind.db
Done.


OrderID,CustomerID,OrderDate,Ranking_orders
26376,ALFKI,2023-10-15 09:15:19,1
15905,ALFKI,2023-10-02 12:20:28,2
13736,ALFKI,2023-09-24 08:20:26,3
17875,ALFKI,2023-09-21 21:43:03,4
17684,ALFKI,2023-08-21 18:01:37,5
12271,ALFKI,2023-08-18 01:47:30,6
15318,ALFKI,2023-07-01 09:32:35,7
18421,ALFKI,2023-05-25 02:46:06,8
24163,ALFKI,2023-04-18 19:18:22,9
15936,ALFKI,2023-03-10 11:35:35,10


### Exercise 2

Calculate a running total of the quantity of orders using window functions.

In [5]:
%%sql

SELECT
    OrderID,
    ProductID,
    Quantity,
    SUM(Quantity) OVER (
        PARTITION BY OrderID
    ) AS Running_total_quantity
FROM
    "Order Details"
LIMIT 50;

 * sqlite:///Northwind.db
Done.


OrderID,ProductID,Quantity,Running_total_quantity
10248,11,12,27
10248,42,10,27
10248,72,5,27
10249,14,9,49
10249,51,40,49
10250,41,10,60
10250,51,35,60
10250,65,15,60
10251,22,6,41
10251,57,15,41


### Exercise 3


Use window functions to find the difference in successive order dates for each customer. **HINT:** The `TIMESTAMPDIFF()` function in MySQL is not available in SQLite. We can use the `julianday()` function to convert the dates to a floating point number and then calculate the difference.

In [6]:
%%sql

SELECT 
    OrderID,
    CustomerID,
    OrderDate,
    LAG(OrderDate, 1) OVER (
        PARTITION BY
            CustomerID
         ORDER BY
            OrderDate
    ) AS Previous_order_date,
    julianday(OrderDate) - julianday(
        LAG(OrderDate, 1) OVER (
            PARTITION BY
                CustomerID
             ORDER BY
                OrderDate
        )
    ) AS Difference_order_date
FROM
    Orders;

 * sqlite:///Northwind.db
Done.


OrderID,CustomerID,OrderDate,Previous_order_date,Difference_order_date
12671,ALFKI,2012-08-22 13:27:12,None,None
12078,ALFKI,2012-08-31 17:02:02,2012-08-22 13:27:12,9.149189814925194
11908,ALFKI,2012-09-07 02:39:28,2012-08-31 17:02:02,6.400995370466262
14208,ALFKI,2012-10-06 12:09:23,2012-09-07 02:39:28,29.395775462966412
19569,ALFKI,2012-12-16 21:52:46,2012-10-06 12:09:23,71.40512731485069
12696,ALFKI,2012-12-18 05:24:55,2012-12-16 21:52:46,1.3139930553734303
11789,ALFKI,2013-01-05 17:27:45,2012-12-18 05:24:55,18.501967592630535
26469,ALFKI,2013-01-06 21:00:22,2013-01-05 17:27:45,1.1476504630409181
16617,ALFKI,2013-02-12 05:51:27,2013-01-06 21:00:22,36.36880787042901
16370,ALFKI,2013-03-25 22:48:40,2013-02-12 05:51:27,41.70640046289191


### Exercise 4

Calculate the moving average of the quantity of the last 3 orders for each product using window functions.

In [7]:
%%sql

SELECT
    OrderID,
    ProductID,
    Quantity,
    AVG(Quantity) OVER (
        PARTITION BY
            ProductID
        ORDER BY
            OrderID
        ROWS BETWEEN
            2
        PRECEDING AND
            0
        FOLLOWING
    ) AS Moving_average_quantity
FROM
    "Order Details"
GROUP BY
    ProductID,
    OrderID
LIMIT 50;

 * sqlite:///Northwind.db
Done.


OrderID,ProductID,Quantity,Moving_average_quantity
10285,1,45,45.0
10294,1,18,31.5
10317,1,20,27.666666666666668
10348,1,15,17.666666666666668
10354,1,12,15.666666666666666
10370,1,15,14.0
10406,1,10,12.333333333333334
10413,1,24,16.333333333333332
10477,1,15,16.333333333333332
10522,1,40,26.333333333333332


## Solutions

### Exercise 1

In [ ]:
%%sql

SELECT 
    OrderID, 
    OrderDate,
    RANK() OVER (ORDER BY OrderDate DESC) as Order_rank
FROM 
    Orders 
WHERE 
    CustomerID = 'ALFKI';

The `RANK()` window function is used here to rank each order of the customer with the ID `'ALFKI'` based on the `OrderDate`. The `DESC` keyword is used so that the most recent order gets the highest rank (i.e. 1).

### Exercise 2

In [ ]:
%%sql

SELECT 
    OrderID, 
    Quantity, 
    SUM(Quantity) OVER (
    ORDER BY OrderID) as RunningTotal 
FROM 
    OrderDetails
GROUP BY 
    OrderID;


The `SUM()` window function is used here to calculate a running total of the quantity of orders. The `ORDER BY` clause inside the `OVER()` clause ensures that the running total is calculated in the order of the `OrderID`. We then group our data by `OrderID`.

### Exercise 3

In [ ]:
%%sql

SELECT 
    CustomerID, 
    OrderDate, 
    LAG(OrderDate, 1) OVER 
        (PARTITION BY CustomerID 
        ORDER BY OrderDate) as PrevOrderDate, 
    julianday(OrderDate)-
    julianday(LAG(OrderDate, 1) OVER
            (PARTITION BY CustomerID 
            ORDER BY OrderDate)) as DateDiff
FROM 
    Orders;

The `LAG()` window function is used twice here, once to get the previous order date for each customer, and again to calculate the difference between the current order date and the previous order date. The `PARTITION BY` clause is used to separate the data into partitions based on the `CustomerID`. 

Since the `TIMESTAMPDIFF()` function in MySQL is not available in SQLite, we use the `julianday()` function to convert the dates to a floating point number, which we can then use for subtraction to find the difference between two dates.

### Exercise 4

In [ ]:
%%sql

SELECT 
    OrderID, 
    ProductID, 
    Quantity,
    AVG(Quantity) OVER (PARTITION BY ProductID ORDER BY OrderID ROWS BETWEEN 2 PRECEDING AND 0 FOLLOWING) as MovingAvgQuantity
FROM 
    OrderDetails
ORDER BY 
    ProductID, 
    OrderID;

The `AVG()` window function is used here to calculate the moving average of `Quantity` for the last 3 orders (the current order and the two preceding orders) for each product. The window is defined using the `PARTITION BY` clause (to segment the data by `ProductID`) and the `ORDER BY` clause (to arrange the data in order of `OrderID`). The `ROWS BETWEEN` clause specifies the size and location of the window – in this case, **the current row and the two rows preceding it.**

#

<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/refs/heads/master/ALX_banners/ALX_Navy.png"  style="width:100px"  ;/>
</div>